# Census NN Experiments

Clean notebook for the 2-layer Census MSGD experiments.

In [ ]:
import os
import pickle
from pathlib import Path

import yaml

from census_nn_final_plots import (
    CensusNNConfig,
    generate_plots,
    maybe_subsample_census_data,
    print_final_accuracies,
    run_msgd_experiments,
    train_baseline_mlp,
)
from census_final_plots import load_and_preprocess_data, perform_clustering, train_baseline_lr
from experiment_io import load_latest_results, select_results
from utils_clustering import create_rankings_from_clusters
from utils_msgd_census_nn import pretrain_partition_models


In [ ]:
CONFIG_PATH = Path("configs/census_nn_good.yaml")
USE_SAVED_RESULTS = False

with open(CONFIG_PATH, "r") as f:
    payload = yaml.safe_load(f)

config = CensusNNConfig(**payload["config"])
num_seeds = int(payload.get("runner", {}).get("num_seeds", 3))
config

In [ ]:
if USE_SAVED_RESULTS:
    saved_payload, saved_dir = load_latest_results(CONFIG_PATH)
    results_dict = select_results(saved_payload)
    print(f"Loaded saved results from {saved_dir}")
    baseline_mlp_acc = None
    baseline_lr_acc = None
else:
    X_train, X_test, y_train, y_test, X_original_train, X_original_test = load_and_preprocess_data()
    X_train, X_test, y_train, y_test, X_original_train, X_original_test = maybe_subsample_census_data(
        X_train,
        X_test,
        y_train,
        y_test,
        X_original_train,
        X_original_test,
        config,
    )
    baseline_lr_acc = train_baseline_lr(X_train, y_train, X_test, y_test, config.reg_lambda)
    baseline_mlp_acc = train_baseline_mlp(X_train, y_train, X_test, y_test, config)
    cluster_labels, kmeans = perform_clustering(X_train, X_original_train, config)
    rankings = create_rankings_from_clusters(X_train, kmeans.cluster_centers_, config.n, cluster_labels)
    init_models = None
    if config.init_method == "partition_pretrain":
        init_models = pretrain_partition_models(X_train, y_train, rankings, config)
    results_dict = run_msgd_experiments(
        X_train,
        y_train,
        X_test,
        y_test,
        rankings,
        init_models,
        config,
        num_seeds=num_seeds,
        baseline_mlp_acc=baseline_mlp_acc,
        baseline_lr_acc=baseline_lr_acc,
    )
    print_final_accuracies(results_dict, baseline_mlp_acc, baseline_lr_acc)


In [ ]:
if USE_SAVED_RESULTS:
    print("Loaded saved results.")
else:
    outcome_name = "census_nn_good_outcome" if config.init_method == "partition_pretrain" else "census_nn_bad_outcome"
    generate_plots(results_dict, X_train, config, baseline_acc=baseline_mlp_acc, prefix=outcome_name)
